# Sentinel-1 bursts → gamma0 RTC VV/VH on a common grid, via OPERA RTC-S1

Same output as `asf_gamma0.ipynb` — one GeoTIFF per acquisition, two named
bands (`gamma0_VV`, `gamma0_VH`), all on a shared grid — but processing **only
the bursts the AOI touches** instead of whole scenes.

**Why a separate notebook.** Burst processing does not go through the same HyP3
job type, and the differences run deep enough to muddy a single notebook:

| | scene notebook (`RTC_GAMMA`) | this one (`OPERA_RTC_S1`) |
| --- | --- | --- |
| method | `submit_rtc_job` | `submit_opera_rtc_s1_job` |
| options | resolution, radiometry, scale, speckle filter, DEM matching | **none at all** |
| resolution | 10, 20 or 30 m, your choice | fixed 30 m |
| granule | the scene name | one **co-pol** burst id, `..._VV_0770-BURST` |
| jobs | 1 per acquisition | 1 per burst |

**When it is worth it.** Only when the AOI is small compared with the 250 km
swath. A whole-scene RTC at 10 m weighs close to 4 GB per date whatever the AOI,
so a handful of bursts is a large saving — but an AOI spanning three sub-swaths
over 150 km can easily need 20 bursts, and then a single scene job is the better
deal. **Cell 5 tells you the count before anything is spent**: compare it with
the one job per date the scene notebook would use, and pick accordingly.

**Two quirks that cost API errors to discover.** OPERA RTC-S1 delivers both
polarisations from a single job, so it must be given the **co-pol** burst only —
submitting the VH twin is rejected with a 400. And it has its own date and
coverage constraints: a granule in the right format can still be refused if the
acquisition falls outside what the OPERA chain covers.

## How to use it

**1. Install the dependencies.**

```
conda install -c conda-forge asf_search hyp3_sdk rasterio geopandas numpy
```

**2. Get a NASA Earthdata Login** (free, <https://urs.earthdata.nasa.gov>) and
store the credentials in a netrc file in your home directory — on Windows,
`C:/Users/<you>/.netrc`:

```
machine urs.earthdata.nasa.gov
login <username>
password <password>
```

`requests` finds that file on its own, by matching the host name.

**3. Provide the inputs** in the parameters cell.

| Parameter | What to give |
| --- | --- |
| `SLC_PATHS` | The acquisitions to process — **the SLC scenes**, not the bursts — as local `.SAFE` / `.zip` paths or as bare granule names. Their acquisition times are what cell 5 uses to find the bursts. A local file additionally lets cell 4 cross-check the burst selection. |
| `AOI` | The area of interest in lon/lat (EPSG:4326): inline WKT, or a `.wkt` / `.geojson` file path. It selects the bursts *and* sets the extent of the outputs. |
| `PIXEL_SIZE` | Resolution of the **output grid** only. OPERA always delivers 30 m, so a finer value here just interpolates. |
| `POLARISATIONS` | The bands to write. Default `["VV", "VH"]` — both come out of the same job. |
| `DOWNLOAD_DIR`, `OUT_DIR` | Where raw products and final GeoTIFFs land. Keep them distinct from other runs: cell 10 searches the tree by date. |
| `JOB_SUFFIX` | Change it to force a genuinely new submission — see the comment in cell 3. |

**4. Run cells 1 to 5 and read them.** Cell 4 must report `OK`, cell 5 lists the
burst granules to be submitted. Nothing has been spent — both only query the free
catalogue.

**5. Run cell 6**, compare the credit balance with the burst count from cell 5.

**6. Run cell 7.** One job per burst, or recovery of the jobs already filed under
`JOB_NAME`. The only cell that can spend credits, and safe to re-run.

**7. Run cell 8, then re-run it every few minutes.** It never blocks: it reports
the statuses and downloads once every job is done.

**8. Run cells 9 to 11.** Purely local, free, replayable.

**After a kernel restart**, run cells 2, 3, 5, 6 and 7 again, then carry on at
cell 8. Nothing is resubmitted and nothing is paid twice.

## What each cell does

| # | What it does | ASF |
| --- | --- | --- |
| 1 | This overview. | — |
| 2 | Imports, and puts the sibling `polygon_to_swaths_bursts` folder on `sys.path`. | — |
| 3 | Every parameter, plus the acquisition times read from the product names. | — |
| 4 | Coverage check: which swaths and bursts of each acquisition the AOI touches, from the local SLC annotations. Falls back to the scene footprint from the catalogue when the file is not available. | catalogue, free |
| 5 | Asks the catalogue for the co-pol burst granules intersecting the AOI, one acquisition at a time. | catalogue, free |
| 6 | Connects to HyP3 through the netrc, prints the credit balance and the `submit_opera_rtc_s1_job` signature. | yes |
| 7 | For each burst, recovers the job already filed under `JOB_NAME` or submits a new one. **The only cell that can spend credits.** | yes |
| 8 | Reports the job statuses without blocking, and downloads and extracts once they are all done. Re-run it until then. | yes |
| 9 | Computes the common output grid: AOI bounding box in UTM, snapped to the pixel size. | — |
| 10 | Regrids every burst raster onto that grid, merges the bursts of a same acquisition, writes one 2-band GeoTIFF per acquisition. | — |
| 11 | Reports the outputs: grid of each file and share of valid pixels. | — |

In [ ]:
# === 2. Imports ===
import inspect
import re
import sys
import zipfile
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

import asf_search as asf
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import Resampling, reproject

# polygon_to_swaths_bursts is a sibling folder: make it importable
TOOLS = Path.cwd().parent / "polygon_to_swaths_bursts"
sys.path.insert(0, str(TOOLS))
from polygon_to_swaths_bursts import get_intersecting_bursts, parse_polygon

In [ ]:
# === 3. Parameters: adapt to your data ===

# The acquisitions to process, named by their SLC product — the SCENES, not the
# bursts. Put as many as you like: every cell loops over this list. ASF works on
# its own copy, nothing is uploaded, so each entry may be a local .SAFE / .zip
# path or just the bare granule name. Cell 5 derives the burst granules from the
# acquisition times; a local file additionally lets cell 4 cross-check them.
SLC_PATHS = [
    r"C:\Users\guigu\Documents\pro_asus\vigisar\data\data_raw\zta8\S1B_IW_SLC__1SDV_20200804T224848_20200804T224915_022779_02B3BE_0770.zip",
]

# Area of interest, lon/lat (EPSG:4326): inline WKT, or a WKT / GeoJSON file path
AOI = "POLYGON ((-63.808599 -24.00883, -62.462762 -24.00883, -62.462762 -25.167657, -63.808599 -25.167657, -63.808599 -24.00883))"

# Where the files land. A relative path resolves against this notebook's folder;
# an absolute one works just as well and is the better habit, since these
# products are heavy and do not belong in the repo. Both are created on the fly.
DOWNLOAD_DIR = "hyp3_downloads/burst"   # raw HyP3 products: zips, then extracted
OUT_DIR = "output/burst"                # the final 2-band GeoTIFFs

# Resolution of the OUTPUT GRID, in metres — and of the output grid ONLY. OPERA
# RTC-S1 always delivers 30 m, so a finer value here merely interpolates: 10 m
# would give files nine times heavier for exactly the same information.
PIXEL_SIZE = 30.0
POLARISATIONS = ["VV", "VH"]

# Free-text label stamped on every job, and the only handle to find them again
# later with hyp3.find_jobs(name=...). It is not unique server-side, and cell 7
# relies on it: any job filed under this exact name in the last two weeks is
# recovered instead of being resubmitted.
#
# HENCE: CHANGE JOB_SUFFIX whenever you want a genuinely new submission — a
# different AOI, a clean retry. Keep the same name and cell 7 will hand you back
# the previous jobs, computed for the previous bursts, without a word.
JOB_SUFFIX = "v1"
JOB_NAME = f"zta8-burst-{JOB_SUFFIX}"

# OPERA RTC-S1 takes no options whatsoever — granule and name, nothing else, as
# cell 6 will show. Radiometry (gamma0), scale, resolution and DEM are all preset
# by the OPERA chain. There is no RTC_OPTIONS dict in this notebook.

TIMESTAMP = re.compile(r"\d{8}T\d{6}")


def acquisition_window(product_name):
    """(start, stop) datetimes read from a Sentinel-1 product name.

    Splitting the name on underscores and indexing by position is unreliable:
    the product-type field is four characters wide, so "SLC_" is padded with an
    underscore while "GRDH" is not, and every later field shifts by one. Picking
    out the two YYYYmmddTHHMMSS tokens works whatever the product type.
    """
    stamps = TIMESTAMP.findall(Path(product_name).stem)
    if len(stamps) < 2:
        raise ValueError(
            f"no pair of acquisition times in {Path(product_name).name!r}"
        )
    return (
        datetime.strptime(stamps[0], "%Y%m%dT%H%M%S"),
        datetime.strptime(stamps[1], "%Y%m%dT%H%M%S"),
    )


# --- How a downloaded raster is paired back with its acquisition --------------
#
# HyP3 does not keep the name of what you submitted, so a file cannot be matched
# to its job or its granule by name. The one token that survives the renaming is
# the acquisition time — and that is what cell 10 matches on.
#
# Here it can only be the DATE, not the full timestamp: each burst starts a few
# seconds into the scene and its product carries the BURST's own time, not the
# scene's, so a second-level key would match nothing.
#
# The failure mode worth remembering: two acquisitions on the same date get
# merged into one output by cell 10, without any error. The check below is there
# to catch exactly that.
STAMP = "%Y%m%d"

# Parsing the names here turns a malformed entry into an immediate error, before
# anything is submitted or paid for
STARTS = [acquisition_window(p)[0] for p in SLC_PATHS]
KEYS = [start.strftime(STAMP) for start in STARTS]

print(f"{len(SLC_PATHS)} acquisition(s), grid at {PIXEL_SIZE:g} m")
print("dates:", ", ".join(sorted(start.strftime("%Y-%m-%d") for start in STARTS)))
print("job name:", JOB_NAME)

if len(set(KEYS)) != len(KEYS):
    print("WARNING: several acquisitions share a date — cell 10 would merge them "
          "into one output")

In [ ]:
# === 4. Which swaths and bursts does the AOI touch? ===
# A cross-check, not a gate: cell 5 selects the bursts by intersection anyway.
# Its value is to show our own reading of the product next to ASF's, so a
# surprising burst count in cell 5 can be spotted immediately.
#
# Reading the bursts needs a local SLC — they only exist there. An entry given as
# a bare granule name falls back to the scene footprint from the ASF catalogue: a
# free query which also confirms that the granule name exists.
aoi_geom = parse_polygon(AOI)

for slc in SLC_PATHS:
    name = Path(slc).stem

    if Path(slc).exists() and "_SLC" in name:
        _, summary = get_intersecting_bursts(slc, AOI, coarse=True)
        if summary:
            detail = ", ".join(f"{sw} {bursts}" for sw, bursts in sorted(summary.items()))
            total = sum(len(bursts) for bursts in summary.values())
            print(f"OK      {name[:58]}\n        {detail}\n        {total} burst(s)")
        else:
            print(f"NO DATA {name[:58]} — the AOI is outside this product")
        continue

    results = asf.granule_search([name])
    if not results:
        print(f"UNKNOWN {name[:58]} — no such granule in the ASF catalogue")
    elif any(parse_polygon(r.geometry).intersects(aoi_geom) for r in results):
        print(f"OK      {name[:58]}\n        AOI inside the scene footprint (catalogue)")
    else:
        print(f"NO DATA {name[:58]} — the AOI is outside this scene")

In [ ]:
# === 5. The burst granules to submit ===
# The catalogue is asked which BURST products intersect the AOI within each
# acquisition's time window — the scene itself is never named. A one-minute
# margin absorbs rounding while staying far narrower than the 6 or 12 days
# between two passes, so only that acquisition can match.
#
# --- Why only the CO-POL bursts are searched ---------------------------------
# This is a naming convention, not a limitation of the processing. OPERA works on
# the BURST — an area on the ground — not on a polarisation channel, and one job
# delivers both VV and VH. The ASF catalogue, on the other hand, splits SLC
# bursts per polarisation, because that is how the source data is stored. So
#
#     S1_006285_IW2_20200804T224848_VV_0770-BURST
#     S1_006285_IW2_20200804T224848_VH_0770-BURST
#
# name the SAME burst. OPERA needs one canonical id for it, and the convention is
# the co-pol one: the VH twin is rejected with a 400, and submitting both would
# be paying twice for one job.
#
# The dual-polarisation output is the documented product spec, not something
# verified here — but cell 10 looks for a *_VH.tif and fails loudly if OPERA
# turns out to deliver only VV, so the first download settles it.
aoi_wkt = parse_polygon(AOI).wkt

GRANULES = []
for slc in SLC_PATHS:
    start, stop = acquisition_window(slc)
    results = asf.search(
        platform=asf.PLATFORM.SENTINEL1,
        processingLevel="BURST",
        intersectsWith=aoi_wkt,
        start=start - timedelta(minutes=1),
        end=stop + timedelta(minutes=1),
        polarization=["VV"],
    )
    # fileID carries the "-BURST" suffix, which is what HyP3 expects here
    found = sorted(r.properties["fileID"] for r in results)
    print(f"{Path(slc).stem[:52]}: {len(found)} burst(s)")
    for granule in found:
        print("   ", granule)
    GRANULES += found

print(f"\n{len(GRANULES)} burst job(s) to submit")
print("Compare with 1 job per acquisition in the scene notebook before spending.")

In [ ]:
# === 6. Connect to HyP3 ===
import hyp3_sdk as sdk

print("hyp3_sdk", sdk.__version__)

# Credentials are read from the netrc file in your home directory
# (C:/Users/<you>/.netrc, or _netrc — requests accepts either):
#
#     machine urs.earthdata.nasa.gov
#     login <earthdata username>
#     password <earthdata password>
#
# Called without arguments, HyP3 lets requests pick them up from there. Pass
# prompt="password" or prompt="token" instead to be asked interactively.
netrc = next(
    (p for p in (Path.home() / ".netrc", Path.home() / "_netrc") if p.exists()), None
)
if netrc is None:
    raise FileNotFoundError(
        f"no .netrc or _netrc found in {Path.home()} — create one as shown above, "
        'or switch to sdk.HyP3(prompt="password")'
    )
print("credentials from", netrc)

hyp3 = sdk.HyP3()

info = hyp3.my_info()
print("user:", info.get("user_id"), "| remaining credits:", info.get("remaining_credits"))

# granule and name, nothing else — that is the whole OPERA RTC-S1 interface
print("\nsubmit_opera_rtc_s1_job", inspect.signature(hyp3.submit_opera_rtc_s1_job))

In [ ]:
# === 7. Submit the missing jobs, recover the ones already filed ===
# The guard works granule by granule, so all three situations behave sensibly:
# re-running after a crash recovers everything, adding acquisitions to SLC_PATHS
# submits only the new bursts, and nothing is ever paid for twice under the same
# JOB_NAME. Failed jobs are ignored, which retries them.
#
# HyP3 keeps job records long after the products expire, so the lookup is limited
# to the last two weeks: an older job cannot be downloaded any more anyway.
recent = datetime.now(timezone.utc) - timedelta(days=14)
existing = hyp3.find_jobs(name=JOB_NAME, start=recent)


def job_granules(job):
    """The granules one job was submitted for."""
    return job.job_parameters.get("granules", [])


wanted = set(GRANULES)
kept = [
    job for job in existing
    if job.status_code != "FAILED" and any(g in wanted for g in job_granules(job))
]
covered = {g for job in kept for g in job_granules(job)}

# Failing loudly here is much cheaper than silently resubmitting everything
if existing and not covered:
    raise RuntimeError(
        f"{len(existing)} job(s) found under {JOB_NAME!r} but their granules could "
        "not be read — refusing to resubmit and risk paying twice. Inspect "
        "existing[0].job_parameters and fix job_granules()."
    )

batch = sdk.Batch(kept)
for granule in GRANULES:
    if granule not in covered:
        batch += hyp3.submit_opera_rtc_s1_job(granule, name=JOB_NAME)

print(f"{JOB_NAME}: {len(kept)} recovered, {len(batch) - len(kept)} submitted")
for job in batch:
    print(f"    {job.status_code:9s} {job.job_id}")

In [ ]:
# === 8. Check the jobs, and download once they are done ===
# Deliberately NOT blocking. hyp3.watch() would hold the kernel for hours, and
# interrupting a blocked kernel is what kills it. Re-run this cell every few
# minutes instead: it reports the statuses, and downloads as soon as all the
# jobs have left PENDING and RUNNING.
#
# HyP3 publishes no progress percentage — only PENDING (queued), RUNNING
# (a worker has it) and SUCCEEDED / FAILED. The elapsed time below is the only
# usable proxy; a burst is far quicker than a whole scene.
batch = hyp3.refresh(batch)
now = datetime.now(timezone.utc)

print(" | ".join(f"{code}: {n}"
                 for code, n in sorted(Counter(j.status_code for j in batch).items())))
for job in batch:
    requested = getattr(job, "request_time", None)
    age = f"{(now - requested).total_seconds() / 60:4.0f} min" if requested else "  ? min"
    print(f"    {job.status_code:9s} {age}  {job.job_id}")

waiting = [job for job in batch if job.status_code in ("PENDING", "RUNNING")]
if waiting:
    print(f"\n{len(waiting)} job(s) still going — re-run this cell in a few minutes")
else:
    failed = [job for job in batch if job.status_code == "FAILED"]
    if failed:
        print(f"\n{len(failed)} job(s) FAILED — re-run cell 7 to retry them")

    succeeded = sdk.Batch([j for j in batch if j.status_code == "SUCCEEDED"])
    download_dir = Path(DOWNLOAD_DIR)
    download_dir.mkdir(parents=True, exist_ok=True)
    zips = succeeded.download_files(location=download_dir)

    for archive in zips:
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(download_dir)
        print("extracted:", Path(archive).name)

In [ ]:
# === 9. The common output grid: AOI bbox in UTM, snapped to PIXEL_SIZE ===
aoi_gs = gpd.GeoSeries([parse_polygon(AOI)], crs="EPSG:4326")
UTM_CRS = aoi_gs.estimate_utm_crs()
minx, miny, maxx, maxy = aoi_gs.to_crs(UTM_CRS).total_bounds

# The grid comes from the AOI alone, never from the images: that is what makes
# every date land on exactly the same pixel centres.
#
# Snapping the origin to a round multiple of PIXEL_SIZE is a separate matter. It
# turns the grid into a canonical lattice attached to (CRS, pixel size) rather
# than to this particular AOI, which buys three things:
#   - editing the AOI later moves the extent by whole pixels instead of sliding
#     the lattice, so new outputs still stack on the old ones;
#   - neighbouring AOIs snapped the same way share the lattice and mosaic
#     without resampling;
#   - other products already on round grids overlay pixel to pixel, which
#     matters for radar/optical fusion.
ULX = np.floor(minx / PIXEL_SIZE) * PIXEL_SIZE
ULY = np.ceil(maxy / PIXEL_SIZE) * PIXEL_SIZE
SIZE_X = int(np.ceil((maxx - ULX) / PIXEL_SIZE))
SIZE_Y = int(np.ceil((ULY - miny) / PIXEL_SIZE))
TRANSFORM = from_origin(ULX, ULY, PIXEL_SIZE, PIXEL_SIZE)

print(f"{UTM_CRS.name} (EPSG:{UTM_CRS.to_epsg()})")
print(f"{SIZE_X} x {SIZE_Y} px at {PIXEL_SIZE} m, upper-left ({ULX}, {ULY})")

In [ ]:
# === 10. Regrid, merge the bursts and write one GeoTIFF per acquisition ===
def find_rtc_bands(key, pol, search_dir):
    """The burst rasters of one acquisition and polarisation.

    Several files here, one per burst — they are merged below. Pairing goes
    through the acquisition date, the only token that survives HyP3's renaming;
    see STAMP in cell 3.
    """
    matches = [
        p for p in Path(search_dir).rglob(f"*_{pol}.tif")
        if key in p.name and "rgb" not in p.name.lower()
    ]
    if not matches:
        raise FileNotFoundError(f"no {pol} raster for {key} under {search_dir}")
    return sorted(matches)


def on_common_grid(src_path):
    """Resample one raster onto the shared grid. Nodata becomes NaN."""
    target = np.full((SIZE_Y, SIZE_X), np.nan, dtype="float32")
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=target,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=TRANSFORM,
            dst_crs=UTM_CRS,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )
    return target


def merge_on_grid(paths):
    """Regrid the bursts and merge them: average where they overlap.

    Each burst raster is already geocoded, so its absolute position is known and
    no relative placement has to be computed. The same averaging handles both
    overlaps: ~1 km in azimuth between consecutive bursts, and 1 to 2 km in range
    between sub-swaths.
    """
    total = np.zeros((SIZE_Y, SIZE_X), dtype="float64")
    count = np.zeros((SIZE_Y, SIZE_X), dtype="uint16")
    for path in paths:
        values = on_common_grid(path)
        valid = np.isfinite(values) & (values > 0)
        total[valid] += values[valid]
        count[valid] += 1
    merged = np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)
    return merged.astype("float32")


outputs = []
for slc, key in zip(SLC_PATHS, KEYS):
    bands = []
    for pol in POLARISATIONS:
        tiles = find_rtc_bands(key, pol, DOWNLOAD_DIR)
        print(f"{key} {pol}: {len(tiles)} burst(s)")
        bands.append(merge_on_grid(tiles))

    out_path = Path(OUT_DIR) / f"{Path(slc).stem}_gamma0.tif"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(
        out_path, "w", driver="GTiff",
        height=SIZE_Y, width=SIZE_X, count=len(POLARISATIONS),
        dtype="float32", crs=UTM_CRS, transform=TRANSFORM, nodata=np.nan,
        compress="deflate", tiled=True,
    ) as dst:
        for index, (pol, band) in enumerate(zip(POLARISATIONS, bands), start=1):
            dst.write(band, index)
            dst.set_band_description(index, f"gamma0_{pol}")

    print("written:", out_path)
    outputs.append(out_path)

In [ ]:
# === 11. Check the outputs: same grid everywhere, enough valid pixels ===
for path in outputs:
    with rasterio.open(path) as src:
        print(path.name)
        print(f"    {src.crs}, {src.width}x{src.height} px, "
              f"pixel {src.transform.a:g} m")
        for index in range(1, src.count + 1):
            data = src.read(index)
            valid = np.isfinite(data) & (data > 0)
            print(f"    {src.descriptions[index - 1]}: "
                  f"{valid.sum() / data.size:.0%} valid pixels")